<a href="https://colab.research.google.com/github/YunaVDaquio/Final-Project-CS2/blob/main/Kamia_Hospital_DB.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import json
# Open the JSON file
with open("/content/hospital.json") as f:
 students = json.load(f)
# Display the JSON content
print(students)

[{'patient_id': 1, 'name': 'John Cruz', 'age': 45, 'gender': 'Male', 'visits': [{'visit_date': '2025-08-12', 'diagnosis': 'Diabetes Type II', 'treatment': 'Metformin 500mg daily', 'doctor': 'Dr. Santos', 'notes': 'Advised low-carb diet and regular exercise.'}, {'visit_date': '2025-09-10', 'diagnosis': 'Diabetes Type II', 'treatment': 'Increased Metformin dosage to 850mg', 'doctor': 'Dr. Santos', 'notes': 'Blood sugar levels still above target.'}]}, {'patient_id': 2, 'name': 'Ana Gomez', 'age': 32, 'gender': 'Female', 'visits': [{'visit_date': '2025-08-15', 'diagnosis': 'Hypertension Stage 1', 'treatment': 'Losartan 50mg daily', 'doctor': 'Dr. Dela Cruz', 'notes': 'Encouraged lifestyle modifications.'}, {'visit_date': '2025-09-05', 'diagnosis': 'Hypertension Stage 1', 'treatment': 'Continue Losartan 50mg', 'doctor': 'Dr. Dela Cruz', 'notes': 'BP improved; maintain current plan.'}]}, {'patient_id': 3, 'name': 'Luis Ramos', 'age': 60, 'gender': 'Male', 'visits': [{'visit_date': '2025-08-1

In [ ]:
!pip install firebase-admin

In [ ]:
import json
from firebase_admin import db

# Load your hospital JSON
with open("/content/hospital.json", "r") as file:
    data = json.load(file)
print("JSON file loaded")

# Reference to 'patients' node in Firebase
ref = db.reference("patients")

# Upload each patient using patient_id as key
for patient in data:  # your JSON is a list, so no ["patients"] needed
    ref.child(str(patient["patient_id"])).set(patient)

print("Data uploaded successfully!")

JSON file loaded
Data uploaded successfully!


In [ ]:
def search_patient(query):
    query = str(query).lower()
    results = []

    for patient in patients:
        if (query == str(patient["patient_id"])) or (query in patient["name"].lower()):
            results.append(patient)

    if results:
        for p in results:
            print(f"ID: {p['patient_id']} | Name: {p['name']} | Age: {p['age']} | Gender: {p['gender']}")
    else:
        print("No matching patient found.")

In [ ]:
import firebase_admin
from firebase_admin import credentials, db

# Initialize Firebase Admin SDK if not already initialized
# Replace 'https://<YOUR_DATABASE_NAME>.firebaseio.com' with your actual Firebase database URL
try:
    cred = credentials.Certificate('/content/firebase_key.json')
    firebase_admin.initialize_app(cred, {
        'databaseURL': 'https://kamia-hospital-db-default-rtdb.asia-southeast1.firebasedatabase.app/'
    })
except ValueError:
    # App already initialized, proceed
    pass

ref = db.reference("patients")

while True:
    print("\n===== HOSPITAL DATABASE MENU =====")
    print("1. Display Patients")
    print("2. Add Patient")
    print("3. Update Patient Info")
    print("4. Add Visit Record")
    print("5. Delete Patient")
    print("6. Exit")

    choice = input("Enter choice: ")

    # DISPLAY PATIENTS
    if choice == "1":
        patients = ref.get()

        if patients:
            print("\nPatient List:")
            for key, patient in patients.items():
                print(f"\nID: {patient['patient_id']}")
                print(f"Name: {patient['name']}")
                print(f"Age: {patient['age']}")
                print(f"Gender: {patient['gender']}")

                print("Visits:")
                for visit in patient.get("visits", []):
                    print(f"  Date: {visit['visit_date']}")
                    print(f"  Diagnosis: {visit['diagnosis']}")
                    print(f"  Treatment: {visit['treatment']}")
                    print(f"  Doctor: {visit['doctor']}")
                    print(f"  Notes: {visit['notes']}")
                    print("  -------------------")
        else:
            print("No patients found.")

    # ADD PATIENT
    elif choice == "2":
        pid = input("Enter Patient ID: ")
        name = input("Enter Name: ")
        age = int(input("Enter Age: "))
        gender = input("Enter Gender: ")

        patient = {
            "patient_id": int(pid),
            "name": name,
            "age": age,
            "gender": gender,
            "visits": []
        }

        ref.child(pid).set(patient)
        print("Patient added successfully!")

    # UPDATE PATIENT INFO
    elif choice == "3":
        pid = input("Enter Patient ID to update: ")
        patient_ref = ref.child(pid)

        patient = patient_ref.get()

        if patient:
            name = input("Enter new name: ")
            age = int(input("Enter new age: "))
            gender = input("Enter new gender: ")

            patient_ref.update({
                "name": name,
                "age": age,
                "gender": gender
            })

            print("Patient updated successfully!")
        else:
            print("Patient not found.")

    # ADD VISIT RECORD
    elif choice == "4":
        pid = input("Enter Patient ID: ")
        patient_ref = ref.child(pid)

        patient = patient_ref.get()

        if patient:
            visit_date = input("Enter Visit Date (YYYY-MM-DD): ")
            diagnosis = input("Enter Diagnosis: ")
            treatment = input("Enter Treatment: ")
            doctor = input("Enter Doctor: ")
            notes = input("Enter Notes: ")

            new_visit = {
                "visit_date": visit_date,
                "diagnosis": diagnosis,
                "treatment": treatment,
                "doctor": doctor,
                "notes": notes
            }

            visits = patient.get("visits", [])
            visits.append(new_visit)

            patient_ref.update({
                "visits": visits
            })

            print("Visit added successfully!")
        else:
            print("Patient not found.")

    # DELETE PATIENT
    elif choice == "5":
        pid = input("Enter Patient ID to delete: ")

        if ref.child(pid).get():
            ref.child(pid).delete()
            print("Patient deleted successfully!")
        else:
            print("Patient not found.")

    # EXIT
    elif choice == "6":
        print("Exiting program...")
        break

    else:
        print("Invalid choice. Try again.")


===== HOSPITAL DATABASE MENU =====
1. Display Patients
2. Add Patient
3. Update Patient Info
4. Add Visit Record
5. Delete Patient
6. Exit
